In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

## Helper Classes

In [ ]:
class ModelComparison:
    def __init__(self, str_uri_rules, str_uri_edges, str_uri_centrality, str_dirname_output):
        self.str_uri_rules = str_uri_rules
        self.str_uri_edges = str_uri_edges
        self.str_uri_centrality = str_uri_centrality
        self.str_dirname_output = str_dirname_output
        self.df_rules = None
        self.df_edges = None
        self.df_centrality = None

    def import_data(self):
        self.df_rules = pd.read_parquet(self.str_uri_rules)
        self.df_edges = pd.read_parquet(self.str_uri_edges)
        self.df_centrality = pd.read_parquet(self.str_uri_centrality)
        # ensure numeric dtypes on rules
        for str_col in ['support', 'confidence', 'lift', 'leverage', 'conviction']:
            if str_col in self.df_rules.columns:
                self.df_rules[str_col] = pd.to_numeric(self.df_rules[str_col], errors='coerce')
        print(f'Association rules loaded: {len(self.df_rules):,}')
        print(f'Network edges loaded: {len(self.df_edges):,}')
        print(f'Network centrality loaded: {len(self.df_centrality):,} items')
        return self

    def plot_top_items_comparison(self, int_top_n=15):
        fig, axes = plt.subplots(1, 2, figsize=(18, 8))
        # association rules: most frequent items in rules
        list_items_rules = []
        for _, row in self.df_rules.iterrows():
            list_items_rules.extend(row['antecedents'].split(', '))
            list_items_rules.extend(row['consequents'].split(', '))
        ser_rule_items = pd.Series(list_items_rules).value_counts().head(int_top_n)
        axes[0].barh(range(len(ser_rule_items)), ser_rule_items.values, color='#4C72B0', edgecolor='black')
        axes[0].set_yticks(range(len(ser_rule_items)))
        axes[0].set_yticklabels(ser_rule_items.index, fontsize=9)
        axes[0].set_title('Top Items in Association Rules', fontsize=14, y=1.02)
        axes[0].set_xlabel('Appearances in Rules')
        axes[0].invert_yaxis()
        # network: top items by weighted degree
        df_top_network = self.df_centrality.head(int_top_n)
        axes[1].barh(range(len(df_top_network)), df_top_network['weighted_degree'].values,
                      color='#DD8452', edgecolor='black')
        axes[1].set_yticks(range(len(df_top_network)))
        axes[1].set_yticklabels(df_top_network['item'].values, fontsize=9)
        axes[1].set_title('Top Items by Network Centrality', fontsize=14, y=1.02)
        axes[1].set_xlabel('Weighted Degree')
        axes[1].invert_yaxis()
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_top_items_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_rule_strength_vs_edge_weight(self):
        # merge: for each rule, find the corresponding edge weight
        list_rows = []
        for _, row in self.df_rules.iterrows():
            str_ante = row['antecedents']
            str_cons = row['consequents']
            # find edge weight for this pair
            df_match = self.df_edges[
                ((self.df_edges['item_a'] == str_ante) & (self.df_edges['item_b'] == str_cons)) |
                ((self.df_edges['item_a'] == str_cons) & (self.df_edges['item_b'] == str_ante))
            ]
            if len(df_match) > 0:
                list_rows.append({
                    'rule': f'{str_ante} → {str_cons}',
                    'lift': row['lift'],
                    'confidence': row['confidence'],
                    'edge_weight': df_match.iloc[0]['weight']
                })
        if len(list_rows) == 0:
            print('No matching edges found for rules.')
            return self
        df_merged = pd.DataFrame(list_rows)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        # lift vs edge weight
        axes[0].scatter(df_merged['edge_weight'], df_merged['lift'],
                        color='#55A868', edgecolors='black', alpha=0.7)
        axes[0].set_title('Association Rule Lift vs Network Edge Weight', fontsize=13, y=1.02)
        axes[0].set_xlabel('Co-occurrence Count (Edge Weight)')
        axes[0].set_ylabel('Lift')
        # confidence vs edge weight
        axes[1].scatter(df_merged['edge_weight'], df_merged['confidence'],
                        color='#55A868', edgecolors='black', alpha=0.7)
        axes[1].set_title('Association Rule Confidence vs Network Edge Weight', fontsize=13, y=1.02)
        axes[1].set_xlabel('Co-occurrence Count (Edge Weight)')
        axes[1].set_ylabel('Confidence')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_rules_vs_network.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def plot_methodology_comparison(self):
        dict_comparison = {
            'Aspect': [
                'Unit of Analysis',
                'Relationship Type',
                'Directionality',
                'Key Metric',
                'Structural Insight',
                'Best For'
            ],
            'Association Rules': [
                'Itemsets → Rules',
                'Conditional probability',
                'Directional (A → B)',
                'Lift, Confidence',
                'Which items predict others',
                'Product recommendations'
            ],
            'Network Analysis': [
                'Items as nodes',
                'Co-occurrence frequency',
                'Undirected (A — B)',
                'Centrality, Community',
                'Item clusters & hubs',
                'Store layout, bundling'
            ]
        }
        df_comparison = pd.DataFrame(dict_comparison)
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.axis('off')
        table = ax.table(
            cellText=df_comparison.values,
            colLabels=df_comparison.columns,
            cellLoc='center',
            loc='center'
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1.2, 1.8)
        for j in range(len(df_comparison.columns)):
            table[0, j].set_facecolor('#4C72B0')
            table[0, j].set_text_props(color='white', fontweight='bold')
        # color method columns
        for i in range(1, len(df_comparison) + 1):
            table[i, 1].set_facecolor('#E8EEF5')
            table[i, 2].set_facecolor('#FCEADD')
        ax.set_title('Methodology Comparison', fontsize=14, y=0.95)
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_methodology_comparison.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def print_comparison(self):
        print('\n=== METHOD COMPARISON ===')
        print(f'\nAssociation Rules (FP-Growth):')
        print(f'  Total rules: {len(self.df_rules):,}')
        if len(self.df_rules) > 0:
            print(f'  Mean lift: {self.df_rules["lift"].mean():.4f}')
            print(f'  Max lift: {self.df_rules["lift"].max():.4f}')
            print(f'  Unique items in rules: {len(set(self.df_rules["antecedents"].tolist() + self.df_rules["consequents"].tolist())):,}')
        print(f'\nNetwork Analysis:')
        print(f'  Nodes (items): {len(self.df_centrality):,}')
        print(f'  Edges (item pairs): {len(self.df_edges):,}')
        print(f'  Mean edge weight: {self.df_edges["weight"].mean():.1f}')
        print(f'  Max edge weight: {self.df_edges["weight"].max():,}')
        print(f'\nKey Difference:')
        print(f'  Association rules are DIRECTIONAL — they tell you P(B|A)')
        print(f'  Network edges are SYMMETRIC — they tell you how often A and B co-occur')
        print(f'  Network analysis also reveals STRUCTURAL patterns (communities, hubs)')
        print(f'  that association rules cannot detect.')
        return self

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = 'comparison'
str_dirname_output = './output'
str_uri_rules = f's3://{str_bucket}/03_association_rules/association_rules.parquet'
str_uri_edges = f's3://{str_bucket}/04_network_analysis/edges.parquet'
str_uri_centrality = f's3://{str_bucket}/04_network_analysis/centrality.parquet'

## Output Directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

Output directory ready: ./output


## Run Comparison

In [ ]:
cls_comparison = ModelComparison(str_uri_rules, str_uri_edges, str_uri_centrality, str_dirname_output)
cls_comparison.import_data()

In [ ]:
cls_comparison.plot_top_items_comparison()

In [ ]:
cls_comparison.plot_rule_strength_vs_edge_weight()

In [ ]:
cls_comparison.plot_methodology_comparison()

In [ ]:
cls_comparison.print_comparison()

## Completion

In [11]:
print('\n=== COMPARISON COMPLETE ===')
print(f'Visualizations saved to: {str_dirname_output}')


=== COMPARISON COMPLETE ===
Visualizations saved to: ./output
